# CSP(1 point)
In the following code, only two of the function usages are essential. One is `is_valid(self, state)`, which checks if the current state is legal; the other is `is_satisfy(self, state)`, which is to checks if the current board meets the win condition. 
The type of state is [], which stores the tuples(a, b), representing the positions of the queens in it.

In test block, for `Solver`,  `n` indicates the size of the test while `method` indicates which bts function will be used(`bts` or `improving_bts`or`min_conflict`); for method `solve`, `render` indicates whether to use a graphical interface representation.

## Question 1: You should complete the function `bts()`. (0.8 points)
You can only use iterative way, not recursive. Using recursion will incur a **20% penalty**. You may add any function you want. 
**Remember to test the case where N=6**

## Question 2: You should complete the function `improving_bts()` or `min_conflict()`. (0.2 points)
For `improving_bts()`, you can use one or more of the following strategies: Minimum Remaining Value, Least constraining value, and Forward checking. You should have a good performance (within 2s) **when N = 16 without GUI**. 

For `min_conflict()`, you should have a good performance  (within 2s) **when N = 100 without GUI**.

### DDL: April.1st
The practice will be checked in this lab class or the next lab class(before **April.1st**) by teachers or SAs. 
#### What will be tested: 
* That you understand every line of your code, not just copy from somewhere 
* That your program compiles correctly
* Correctness of the program logic 
* That the result is obtained in a reasonable time 

#### Grading: 
* Submissions in this lab class: 1.1 points.
* Submissions on time: 1 point.


In [46]:
import numpy as np
import time

def my_range(start, end):
    if start <= end:
        return range(start, end + 1)
    else:
        return range(start, end - 1, -1)


class Problem:
    char_mapping = ('·', 'Q')

    def __init__(self, n=4):
        self.N = n

    def is_valid(self, state):
        """
        check whether the state satisfies the condition or not.
        : param state: list of points (in (row id, col id) tuple form)
        : return: bool value of valid or not
        """
        board = self.get_board(state)
        res = True
        for point in state:
            i, j = point
            condition1 = board[:, j].sum() <= 1
            condition2 = board[i, :].sum() <= 1
            condition3 = self.pos_slant_condition(board, point)
            condition4 = self.neg_slant_condition(board, point)
            res = res and condition1 and condition2 and condition3 and condition4
            if not res:
                break
        return res

    def is_satisfy(self, state):
        return self.is_valid(state) and len(state) == self.N

    def pos_slant_condition(self, board, point):
        i, j = point
        tmp = min(self.N - i - 1, j)
        start = (i + tmp, j - tmp)
        tmp = min(i, self.N - j - 1)
        end = (i - tmp,  j + tmp)
        rows = my_range(start[0], end[0])
        cols = my_range(start[1], end[1])
        return board[rows, cols].sum() <= 1

    def neg_slant_condition(self, board, point):
        i, j = point
        tmp = min(i, j)
        start = (i - tmp, j - tmp)
        tmp = min(self.N - i - 1, self.N - j - 1)
        end = (i + tmp, j + tmp)
        rows = my_range(start[0], end[0])
        cols = my_range(start[1], end[1])
        return board[rows, cols].sum() <= 1

    def get_board(self, state):
        board = np.zeros([self.N, self.N], dtype=int)
        for point in state:
            board[point] = 1
        return board

    def print_state(self, state):
        board = self.get_board(state)
        print('_' * (2 * self.N + 1))
        for row in board:
            for item in row:
                print(f'|{Problem.char_mapping[item]}', end='')
            print('|')
        print('-' * (2 * self.N + 1))

In [47]:
class Solver:
    def __init__(self, n, method):
        self.N = n
        self.problem = Problem(n)
        self.method = method

    def solve(self, render=True):
        if render:
            import pygame
            w, h = 90 * self.n + 10, 90 * self.N + 10
            screen = pygame.display.set_mode((w, h))
            screen.fill('white')
            action_generator = self.method(self.problem)
            clk = pygame.time.Clock()
            queen_img = pygame.image.load('./queen.png')
            while True:
                for event in pygame.event.get():
                    if event == pygame.QUIT:
                        exit()
                try:
                    actions = next(action_generator)
                    screen.fill('white')
                    for i in range(self.n + 1):
                        pygame.draw.rect(screen, 'black', (i * 90, 0, 10, h))
                        pygame.draw.rect(screen, 'black', (0, i * 90, w, 10))
                    for action in actions:
                        i, j = action
                        screen.blit(queen_img, (10 + 90 * j, 10 + 90 * i))
                    pygame.display.flip()
                except StopIteration:
                    pass
                clk.tick(5)
            pass
        else:
            start_time = time.time()
            for actions in self.method(self.problem):
                pass
            self.problem.print_state(actions)
            print(time.time() - start_time)

# BTS: Backtracking search

In [48]:
def bts(problem):
    N = problem.N
    action_stack = []
    nxt = [0] * N

    while(True):
        i = len(action_stack)
        if i == N:
            yield action_stack.copy()
            return
        done = False
        for j in range(nxt[i], N):
            cands = action_stack + [(i, j)]
            if problem.is_valid(cands):
                action_stack.append((i, j))
                nxt[i] = j + 1
                done = True
                if i + 1 < N:
                    nxt[i + 1] = 0
                yield action_stack.copy()
                break
        if not done:
            nxt[i] = 0
            if i > 0:
                action_stack.pop()
                yield action_stack.copy()                               

# Improving BTS 
* Which variable should be assigned next?
* In what order should its values be tried?
* Can we detect inevitable failure early?

## Minimum Remaining Value
Choose the variable with the fewest legal values in its domain
## Least constraining value
Given a variable, choose the least constraining value: the one that rules out the fewest values in the remaining variables
## Forward checking
Keep track of remaining legal values for the unassigned variables. Terminate when any variable has no legal values.

In [49]:
def improving_bts(problem):
    N = problem.N
    action_stack = []
    pos = {}
    cols = set()
    diag1 = set()
    diag2 = set()
    stack = []

    def Legals(i):
        return [
            j for j in range(N) if 
                        j not in cols
                and i - j not in diag1
                and i + j not in diag2
        ]

    def Pick():
        cands = [i for i in range(N) if i not in pos]
        if not cands:
            return -1, []
        i = min(cands, key=lambda x: len(Legals(x)))
        return i, Legals(i)

    def Sort(i, cands):
        _cands = [j for j in range(N) if j not in pos and j != i] # leftover rows

        def Rank(j):
            cols.add(j)
            diag1.add(i - j)
            diag2.add(i + j)
            s = sum(len(Legals(r)) for r in _cands)
            cols.remove(j)
            diag1.remove(i - j)
            diag2.remove(i + j)
            return s

        return sorted(cands, key=Rank, reverse=True)

    def Check():
        return all(len(Legals(i)) > 0 for i in range(N) if i not in pos)

    while(True):
        if len(action_stack) == N:
            yield action_stack.copy()
            return

        if len(stack) == len(action_stack):
            i, cands = Pick()
            if i == -1:
                return
            stack.append({"i": i, "cands": Sort(i, cands), "k": 0})

        cur = stack[len(action_stack)]
        i = cur["i"]
        cands = cur["cands"]

        if cur["k"] == len(cands):
            stack.pop()
            if not action_stack:
                return
            i, j = action_stack.pop()
            pos.pop(i)
            cols.remove(j)
            diag1.remove(i - j)
            diag2.remove(i + j)
            yield action_stack.copy()
            continue

        j = cands[cur["k"]]
        cur["k"] += 1

        pos[i] = j
        cols.add(j)
        diag1.add(i - j)
        diag2.add(i + j)
        action_stack.append((i, j))
        yield action_stack.copy()

        if not Check():
            action_stack.pop()
            pos.pop(i)
            cols.remove(j)
            diag1.remove(i - j)
            diag2.remove(i + j)
            yield action_stack.copy()

## Local search for CSP(min-conflict algorithm)

* First generate a complete assignment for all variables (this set of assignments may conflict)

* Repeat the following steps until there are no conflicts:

    * Randomly Select a variable that causes conflicts

    * Reassign the value of this variable to another value that with the least constraint onflicts with other variables

In [50]:
def min_conflict(problem):
    N = problem.N
    rng = np.random.default_rng()
    lim = max(1000, 50 * N)

    while(True):
        cur = rng.integers(0, N, size=N)
        col_cnt = np.zeros(N, dtype=int)
        diag1_cnt = np.zeros(2 * N - 1, dtype=int)
        diag2_cnt = np.zeros(2 * N - 1, dtype=int)

        for i, j in enumerate(cur):
            col_cnt[j] += 1
            diag1_cnt[i - j + N - 1] += 1
            diag2_cnt[i + j] += 1

        def Conflicts(i, j):
            return col_cnt[j] + diag1_cnt[i - j + N - 1] + diag2_cnt[i + j] - 3

        action_stack = [(i, int(j)) for i, j in enumerate(cur)]
        yield action_stack.copy()

        for _ in range(lim):
            clashed = [i for i, j in enumerate(cur) if Conflicts(i, j) > 0]

            if not clashed:
                yield action_stack.copy()
                return

            i = int(rng.choice(clashed))
            old = int(cur[i])

            col_cnt[old] -= 1
            diag1_cnt[i - old + N - 1] -= 1
            diag2_cnt[i + old] -= 1

            cands = [Conflicts(i, j) for j in range(N)]
            best = min(cands)
            j = int(rng.choice([j for j in range(N) if cands[j] == best]))

            cur[i] = j
            col_cnt[j] += 1
            diag1_cnt[i - j + N - 1] += 1
            diag2_cnt[i + j] += 1
            action_stack[i] = (i, j)
            yield action_stack.copy()

## test block

In [51]:
Solver(n=6, method=bts).solve(render=False)

_____________
|·|Q|·|·|·|·|
|·|·|·|Q|·|·|
|·|·|·|·|·|Q|
|Q|·|·|·|·|·|
|·|·|Q|·|·|·|
|·|·|·|·|Q|·|
-------------
0.0050084590911865234


In [52]:
Solver(n=16, method=improving_bts).solve(render=False)

_________________________________
|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|
|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|
|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|
|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|
|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|
|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|
|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|
---------------------------------
0.008000612258911133


In [53]:
Solver(n=100, method=min_conflict).solve(render=False)

_________________________________________________________________________________________________________________________________________________________________________________________________________
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·